In [5]:
import pandas as pd
import os
import json

def get_top_features_with_samples(folder, min_score=1, top_k_samples=50):
    """Get samples for all validated features with score >= min_score"""
    
    # Load validated features
    labels_path = os.path.join(folder, "feature_labels_validated/neulab-claude-sonnet-4-20250514.json")
    with open(labels_path, "r") as f:
        validated_features = json.load(f)
    
    # Load activations, embeddings, and word samples
    act_df = pd.read_csv(os.path.join(folder, "top-50_activations.csv"))
    emb_df = pd.read_csv(os.path.join(folder, "feature_sample_centroid-metrics.csv"))
    with open(os.path.join(folder, "top-50_words_in_context.json"), "r") as f:
        word_samples = json.load(f)
    
    results = {}
    
    # Filter features by score
    for feature_id, feature_data in validated_features.items():
        score = int(feature_data.get("Score", 0))
        if score < min_score:
            continue
        
        # Get activations for this feature
        feat_acts = act_df[act_df["feature"].astype(str) == str(feature_id)].copy()
        
        # Add cosine similarity
        feat_emb = emb_df[emb_df["feature"].astype(str) == str(feature_id)]["sample_centroid_cos_sim"].values
        if len(feat_emb) > 0:
            feat_acts["cos_sim"] = feat_emb[:len(feat_acts)]
            # Sort by cosine similarity
            feat_acts = feat_acts.sort_values("cos_sim", ascending=False)
        
        # Extract samples
        samples = []
        for _, row in feat_acts.head(top_k_samples).iterrows():
            wid = str(row["word_id"])
            word_info = word_samples.get(wid, {"before": "", "word": "", "after": ""})
            
            samples.append({
                "word_id": wid,
                "activation": float(row["act_value"]),
                "cos_sim": float(row.get("cos_sim", 0)),
                "context": f"{word_info.get('before', '')} <{word_info.get('word', '')}> {word_info.get('after', '')}"
            })
        
        results[feature_id] = {
            "description": feature_data.get("Description", ""),
            "score": score,
            "winning_rank": feature_data.get("Winning Rank"),
            "model": feature_data.get("Model"),
            "samples": samples
        }
    
    return results



In [8]:
# Example usage:
folder = "/data/user_data/nsrikant/bbox_data/sae_outputs/sae_outputs_olmo3_7b_unigram_power_0_5/n_moreearly_olmo3_7b_unigram_seed=42_ofw=0.7_N=3000_k=200_lp=None"
top_features = get_top_features_with_samples(folder, min_score=1, top_k_samples=10)

# Print samples for a specific feature
for fid, data in top_features.items():
    print(f"\nFeature {fid}: {data['description']}")
    print(f"Score: {data['score']}, Model: {data['model']}")
    for i, sample in enumerate(data['samples'][:10], 1):
        print(f"  {i}. {sample['context']} (activation: {sample['activation']:.3f})")


Feature 801: Commas used as punctuation to separate clauses or phrases in written text, particularly after introductory elements or to set off parenthetical information.
Score: 3, Model: olmo-3-7b-stage1-step73000
  1.  inhibit the production of meat? According to the Minister <,>  these include low investment levels, undeveloped  infrastructure, (activation: 1.691)
  2.  on Caetanya's book, The Restoration of African Greatness <,>  in which the argument has been advanced that the ancient (activation: 1.782)
  3.  his prerogative to rejoin Kanu, but, quite obviously <,>  that was not what he meant by saying he would (activation: 1.599)
  4.  the displaced people return to their homes.	However <,>  Mr Kerario did not go further than saying that the (activation: 1.749)
  5.  in the first place.	Why, for instance <,>  did the Bank provide political patronage to these institutions, (activation: 1.687)
  6.  husbandry.	Owing to the increasing pressure on land <,>  the government will have 